<a href="https://colab.research.google.com/github/July230/CopyCode/blob/develop/LecturaDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Librerías

In [ ]:
import numpy as np
import pandas as pd
import io
import os
import re
import tensorflow as tf
from collections import Counter, defaultdict
from google.colab import files
from concurrent.futures import ThreadPoolExecutor
from transformers import BertTokenizer, BertModel
from t.text import BertTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dataframes

In [ ]:
parent_directory = "/content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio"
dataset_directory = os.path.join(parent_directory, "dataset")

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_3096/2135108706.py:1: SyntaxWarning: invalid escape sequence '\ '
  parent_directory = "/content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio"


In [ ]:
%cd {dataset_directory}
%ls

/content/drive/.shortcut-targets-by-id/1Xut7nWIe-55SLuAofdhgyG3ekr1fIvsa/IA y Compiladores/Reto_plagio/dataset
checkpoint_m1/  dev/     test/                train/     train_source.parquet
dataset/        dev.csv  tokens_official.csv  train.csv  unlabeled_test.csv


In [ ]:
df_train = pd.read_csv("./train.csv")

In [ ]:
df_validation = pd.read_csv("./dev.csv")

In [ ]:
df_test = pd.read_csv("./unlabeled_test.csv")

In [ ]:
df_all = pd.concat([df_train, df_validation, df_test])

In [ ]:
df_all

,uid,pid
0,415.0,27909
1,415.0,55938
2,415.0,90936
3,415.0,56858
4,415.0,9447
...,...,...
24995,NaN,54090
24996,NaN,62563
24997,NaN,45920
24998,NaN,81447


# Convertir archivos a dataframe

In [ ]:
def build_source_dataframe(path, metadata_df):
  """
  Copies the source code to a column of a dataframe

  Parameters
  ----------

  """
  records = []

  for idx, row in enumerate(metadata_df.itertuples(index=False)):

      pid = str(row.pid)

      file_path = os.path.join(path, pid)

      try:
          with open(file_path, "r", encoding="utf-8") as file:
              source_code = file.read()

          records.append({
              "uid": row.uid,
              "pid": pid,
              "source_code": source_code,
              "num_chars": len(source_code),
              "num_lines": source_code.count("\n") + 1
          })

      except Exception as e:
          print(f"Error leyendo {pid}: {e}")

      if idx % 100 == 0:
          print(f"Procesados {idx} archivos")

  return pd.DataFrame(records)


In [ ]:
train_path = os.path.join(dataset_directory, "train")
dev_path = os.path.join(dataset_directory, "dev")
test_path = os.path.join(dataset_directory, "test")

## Train

In [ ]:
%ls {dataset_directory}

checkpoint_m1/  dev/     test/                train/     train_source.parquet
dataset/        dev.csv  tokens_official.csv  train.csv  unlabeled_test.csv


In [ ]:
"""
train_source = build_source_dataframe(
    path=train_path,
    metadata_df=df_train
)
"""

Procesados 0 archivos
Procesados 100 archivos
Procesados 200 archivos
Procesados 300 archivos
Procesados 400 archivos
Procesados 500 archivos
Procesados 600 archivos
Procesados 700 archivos
Procesados 800 archivos
Procesados 900 archivos
Procesados 1000 archivos
Procesados 1100 archivos
Procesados 1200 archivos
Procesados 1300 archivos
Procesados 1400 archivos
Procesados 1500 archivos
Procesados 1600 archivos
Procesados 1700 archivos
Procesados 1800 archivos
Procesados 1900 archivos
Procesados 2000 archivos
Procesados 2100 archivos
Procesados 2200 archivos
Procesados 2300 archivos
Procesados 2400 archivos
Procesados 2500 archivos
Procesados 2600 archivos
Procesados 2700 archivos
Procesados 2800 archivos
Procesados 2900 archivos
Procesados 3000 archivos
Procesados 3100 archivos
Procesados 3200 archivos
Procesados 3300 archivos
Procesados 3400 archivos
Procesados 3500 archivos
Procesados 3600 archivos
Procesados 3700 archivos
Procesados 3800 archivos
Procesados 3900 archivos
Procesados 4

In [ ]:
output_path = os.path.join(dataset_directory, "train_source.parquet")

In [ ]:
%cd {dataset_directory}

/content/drive/.shortcut-targets-by-id/1Xut7nWIe-55SLuAofdhgyG3ekr1fIvsa/IA y Compiladores/Reto_plagio/dataset


In [ ]:
%ls {output_path}

'/content/drive/MyDrive/IA y Compiladores/Reto_plagio/dataset/train_source.parquet'


In [ ]:
"""
train_source.to_parquet(
    "train_source.parquet",
    index=False
)

print(f"Guardado en: {output_path}")
"""

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet


In [ ]:
train_source = pd.read_parquet("train_source.parquet")

In [ ]:
train_source

,uid,pid,source_code,num_chars,num_lines
0,415,27909,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,4144,158
1,415,55938,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,2840,90
2,415,90936,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,2762,94
3,415,56858,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,3094,105
4,415,9447,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,2603,83
...,...,...,...,...,...
49995,276,62598,#include <iostream>\n#include<bits/stdc++.h>\n...,1950,103
49996,276,56580,#define _CRT_SECURE_NO_WARNINGS 1\n#include <i...,1834,93
49997,276,52656,#define _CRT_SECURE_NO_WARNINGS 1\n#include <v...,2235,100
49998,276,3563,#define _CRT_SECURE_NO_WARNINGS 1\n#include <v...,2225,95


Hacer la lectura de forma paralela

In [ ]:
def process_chunk(path, chunk_df, worker_id):

    records = []

    total = len(chunk_df)

    for idx, row in enumerate(chunk_df.itertuples(index=False), start=1):

        pid = str(row.pid)

        file_path = os.path.join(path, pid)

        try:
            with open(file_path, "r", encoding="utf-8") as f:

                source_code = f.read()

            records.append({
                "uid": row.uid,
                "pid": pid,
                "source_code": source_code,
                "num_chars": len(source_code),
                "num_lines": source_code.count("\n") + 1
            })

        except Exception as e:
            print(f"Error en {pid}: {e}")

        if idx % 100 == 0:
            print(
                f"[Worker {worker_id}] "
                f"{idx}/{total} archivos procesados"
            )

    return pd.DataFrame(records)


def build_source_dataframe_parallel(path, metadata_df, workers=5):

    chunks = np.array_split(metadata_df, workers)

    with ThreadPoolExecutor(max_workers=workers) as executor:

        futures = [
            executor.submit(process_chunk, path, chunk, worker_id)
            for worker_id, chunk in enumerate(chunks, start=1)
        ]

        dfs = [future.result() for future in futures]

    return pd.concat(dfs, ignore_index=True)

In [ ]:
%cd {dataset_directory}

/content/drive/.shortcut-targets-by-id/1Xut7nWIe-55SLuAofdhgyG3ekr1fIvsa/IA y Compiladores/Reto_plagio/dataset


In [ ]:
!ls

checkpoint_m1  dev	test		     train	train_source.parquet
dataset        dev.csv	tokens_official.csv  train.csv	unlabeled_test.csv


## Dev/Validation

In [ ]:
dev_source = build_source_dataframe_parallel(
    path="./dev",
    metadata_df=df_validation,
    workers=5
)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[Worker 1] 100/5000 archivos procesados
[Worker 2] 100/5000 archivos procesados
[Worker 3] 100/5000 archivos procesados
[Worker 5] 100/5000 archivos procesados
[Worker 4] 100/5000 archivos procesados
[Worker 1] 200/5000 archivos procesados
[Worker 2] 200/5000 archivos procesados
[Worker 3] 200/5000 archivos procesados
[Worker 5] 200/5000 archivos procesados
[Worker 4] 200/5000 archivos procesados
[Worker 1] 300/5000 archivos procesados
[Worker 2] 300/5000 archivos procesados
[Worker 3] 300/5000 archivos procesados
[Worker 4] 300/5000 archivos procesados
[Worker 5] 300/5000 archivos procesados
[Worker 5] 400/5000 archivos procesados
[Worker 1] 400/5000 archivos procesados
[Worker 3] 400/5000 archivos procesados
[Worker 2] 400/5000 archivos procesados
[Worker 4] 400/5000 archivos procesados
[Worker 1] 500/5000 archivos procesados
[Worker 3] 500/5000 archivos procesados
[Worker 4] 500/5000 archivos procesados
[Worker 2] 500/5000 archivos procesados
[Worker 5] 500/5000 archivos procesados


In [ ]:
dev_source

,uid,pid,source_code,num_chars,num_lines
0,415,12180,/*{{{*/\n#include <bits/stdc++.h>\n#define SZ(...,2656,80
1,415,67901,/*{{{*/\n#include<cstdio>\n#include<cstdlib>\n...,2629,83
2,415,76245,/*{{{*/\n#include <bits/stdc++.h>\n#define SZ(...,3280,111
3,415,70953,/*{{{*/\n#include <bits/stdc++.h>\n#define SZ(...,2586,83
4,415,41256,/*{{{*/\n#include <bits/stdc++.h>\n#define SZ(...,3395,117
...,...,...,...,...,...
24995,276,57950,#define _CRT_SECURE_NO_WARNINGS 1\n#include <v...,1664,77
24996,276,53982,#include<iostream>\n#include<algorithm>\n#incl...,685,32
24997,276,88752,#define _CRT_SECURE_NO_WARNINGS 1\n#include <v...,2664,104
24998,276,81705,#define _CRT_SECURE_NO_WARNINGS 1\n#include <v...,1344,54


In [ ]:
%cd {dataset_directory}

/content/drive/.shortcut-targets-by-id/1Xut7nWIe-55SLuAofdhgyG3ekr1fIvsa/IA y Compiladores/Reto_plagio/dataset


In [ ]:
dev_source.to_parquet(
    "dev_source.parquet",
    index=False
)

print(f"Guardado en: {output_path}")

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet


## Test

In [ ]:
%cd {dataset_directory}

/content/drive/.shortcut-targets-by-id/1Xut7nWIe-55SLuAofdhgyG3ekr1fIvsa/IA y Compiladores/Reto_plagio/dataset


In [ ]:
!ls

checkpoint_m1  dev.csv		   tokens_official.csv	train_source.parquet
dataset        dev_source.parquet  train		unlabeled_test.csv
dev	       test		   train.csv


In [ ]:
test_source = build_source_dataframe_parallel(
    path="./test",
    metadata_df=df_test,
    workers=10
)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


[Worker 2] 100/2500 archivos procesados
[Worker 1] 100/2500 archivos procesados
[Worker 3] 100/2500 archivos procesados
[Worker 4] 100/2500 archivos procesados
[Worker 5] 100/2500 archivos procesados
[Worker 9] 100/2500 archivos procesados
[Worker 6] 100/2500 archivos procesados
[Worker 7] 100/2500 archivos procesados
[Worker 8] 100/2500 archivos procesados
[Worker 10] 100/2500 archivos procesados
[Worker 2] 200/2500 archivos procesados
[Worker 3] 200/2500 archivos procesados
[Worker 4] 200/2500 archivos procesados
[Worker 5] 200/2500 archivos procesados
[Worker 9] 200/2500 archivos procesados
[Worker 1] 200/2500 archivos procesados
[Worker 6] 200/2500 archivos procesados
[Worker 10] 200/2500 archivos procesados
[Worker 8] 200/2500 archivos procesados
[Worker 7] 200/2500 archivos procesados
[Worker 3] 4800/5000 archivos procesados
[Worker 4] 4800/5000 archivos procesados
[Worker 2] 300/2500 archivos procesados
[Worker 4] 300/2500 archivos procesados
[Worker 3] 300/2500 archivos procesa

In [ ]:
test_source

,uid,pid,source_code,num_chars,num_lines
0,NaN,34841,"///If I see that thing, I'ma squash it\n///and...",634,33
1,NaN,68320,/*input\n1\n1\n\n*/\n \n/*\n ...,3100,125
2,NaN,72734,#include <algorithm>\n#include <iostream>\n#in...,1576,81
3,NaN,55803,//HASHEMESHOON hastam ;)\n#include <iostream>\...,682,31
4,NaN,96166,#include <functional>\n#include <algorithm>\n#...,3410,166
...,...,...,...,...,...
24995,NaN,54090,#include <iostream>\nusing namespace std;\n\nl...,194,14
24996,NaN,62563,#include <bits/stdc++.h>\nusing namespace std;...,230,16
24997,NaN,45920,#include <bits/stdc++.h>\n#define pb push_back...,2021,80
24998,NaN,81447,#include <bits/stdc++.h>\n#define int __int64\...,550,27


In [ ]:
test_source.to_parquet(
    "test_source.parquet",
    index=False
)

print(f"Guardado en: {output_path}")

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet


# Obtener tokens

In [ ]:
tokenizer_bert = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

def split_tokens(text, chunk_size=400):
    tokens = tokenizer_bert.tokenize(text)
    return [
        tokens[i:i + chunk_size]
        for i in range(0, len(tokens), chunk_size)
    ]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_source_code(text):

  if not isinstance(text, str) or len(text.strip()) == 0:
        return pd.DataFrame()

  encoded = tokenizer_bert(
      text,
      max_length=512,
      truncation=True,
      return_overflowing_tokens=True,
      padding=False
  )

  rows = []

  for chunk_id, token_ids in enumerate(encoded["input_ids"]):

    rows.append({
        "chunk_id": chunk_id,
        "token_ids": token_ids,
        "num_tokens": len(token_ids)
    })

  return pd.DataFrame(rows)

def build_tokens_dataframe(source_df):
  all_chunks = []

  total = len(source_df)

  for idx, row in enumerate(source_df.itertuples(index=False),start=1):

      chunks_df = tokenize_source_code(row.source_code)

      if chunks_df.empty:
        continue

      chunks_df["uid"] = row.uid
      chunks_df["pid"] = row.pid

      all_chunks.append(chunks_df)

      if idx % 100 == 0:
          print(f"{idx}/{total} archivos tokenizados")

  return pd.concat(all_chunks, ignore_index=True)

In [ ]:
train_tokens = build_tokens_dataframe(train_source)

100/50000 archivos tokenizados
200/50000 archivos tokenizados
300/50000 archivos tokenizados
400/50000 archivos tokenizados
500/50000 archivos tokenizados
600/50000 archivos tokenizados
700/50000 archivos tokenizados
800/50000 archivos tokenizados
900/50000 archivos tokenizados
1000/50000 archivos tokenizados
1100/50000 archivos tokenizados
1200/50000 archivos tokenizados
1300/50000 archivos tokenizados
1400/50000 archivos tokenizados
1500/50000 archivos tokenizados
1600/50000 archivos tokenizados
1700/50000 archivos tokenizados
1800/50000 archivos tokenizados
1900/50000 archivos tokenizados
2000/50000 archivos tokenizados
2100/50000 archivos tokenizados
2200/50000 archivos tokenizados
2300/50000 archivos tokenizados
2400/50000 archivos tokenizados
2500/50000 archivos tokenizados
2600/50000 archivos tokenizados
2700/50000 archivos tokenizados
2800/50000 archivos tokenizados
2900/50000 archivos tokenizados
3000/50000 archivos tokenizados
3100/50000 archivos tokenizados
3200/50000 archiv

In [ ]:
train_tokens

,chunk_id,token_ids,num_tokens,uid,pid
0,0,"[101, 1013, 1008, 1063, 1063, 1063, 1008, 1013...",512,415,27909
1,1,"[101, 21189, 12879, 3940, 1026, 20014, 1010, 2...",512,415,27909
2,2,"[101, 11675, 1059, 1006, 9530, 3367, 1056, 100...",512,415,27909
3,3,"[101, 1045, 1011, 1011, 1007, 1063, 16360, 100...",407,415,27909
4,0,"[101, 1013, 1008, 1063, 1063, 1063, 1008, 1013...",512,415,55938
...,...,...,...,...,...
95386,1,"[101, 8889, 8889, 8889, 8889, 8889, 2692, 1001...",507,276,3563
95387,0,"[101, 1001, 9375, 1035, 13675, 2102, 1035, 585...",512,276,75845
95388,1,"[101, 1011, 6694, 8889, 8889, 8889, 8889, 8889...",512,276,75845
95389,2,"[101, 1006, 1048, 2487, 1007, 1025, 16596, 100...",512,276,75845


In [ ]:
train_tokens.to_csv("train_tokens.csv", index=False)

print(f"Guardado en: {output_path}")

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet


In [ ]:
dev_tokens = build_tokens_dataframe(dev_source)

100/25000 archivos tokenizados
200/25000 archivos tokenizados
300/25000 archivos tokenizados
400/25000 archivos tokenizados
500/25000 archivos tokenizados
600/25000 archivos tokenizados
700/25000 archivos tokenizados
800/25000 archivos tokenizados
900/25000 archivos tokenizados
1000/25000 archivos tokenizados
1100/25000 archivos tokenizados
1200/25000 archivos tokenizados
1300/25000 archivos tokenizados
1400/25000 archivos tokenizados
1500/25000 archivos tokenizados
1600/25000 archivos tokenizados
1700/25000 archivos tokenizados
1800/25000 archivos tokenizados
1900/25000 archivos tokenizados
2000/25000 archivos tokenizados
2100/25000 archivos tokenizados
2200/25000 archivos tokenizados
2300/25000 archivos tokenizados
2400/25000 archivos tokenizados
2500/25000 archivos tokenizados
2600/25000 archivos tokenizados
2700/25000 archivos tokenizados
2800/25000 archivos tokenizados
2900/25000 archivos tokenizados
3000/25000 archivos tokenizados
3100/25000 archivos tokenizados
3200/25000 archiv

In [ ]:
dev_tokens

,chunk_id,token_ids,num_tokens,uid,pid
0,0,"[101, 1013, 1008, 1063, 1063, 1063, 1008, 1013...",512,415,12180
1,1,"[101, 1028, 1060, 1025, 1065, 11675, 1035, 105...",512,415,12180
2,2,"[101, 1007, 1063, 6140, 2546, 1006, 1000, 1001...",322,415,12180
3,0,"[101, 1013, 1008, 1063, 1063, 1063, 1008, 1013...",512,415,67901
4,1,"[101, 21189, 12879, 3940, 1026, 20014, 1010, 2...",512,415,67901
...,...,...,...,...,...
47349,1,"[101, 1011, 6694, 8889, 8889, 8889, 8889, 8889...",512,276,88752
47350,2,"[101, 1065, 2065, 1006, 999, 5210, 1007, 1063,...",210,276,88752
47351,0,"[101, 1001, 9375, 1035, 13675, 2102, 1035, 585...",512,276,81705
47352,1,"[101, 1045, 1033, 1012, 2117, 1027, 1045, 1025...",85,276,81705


In [ ]:
dev_tokens.to_csv("dev_tokens.csv", index=False)

print(f"Guardado en: {output_path}")

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet


In [ ]:
test_tokens = build_tokens_dataframe(test_source)

100/25000 archivos tokenizados
200/25000 archivos tokenizados
300/25000 archivos tokenizados
400/25000 archivos tokenizados
500/25000 archivos tokenizados
600/25000 archivos tokenizados
700/25000 archivos tokenizados
800/25000 archivos tokenizados
900/25000 archivos tokenizados
1000/25000 archivos tokenizados
1100/25000 archivos tokenizados
1200/25000 archivos tokenizados
1300/25000 archivos tokenizados
1400/25000 archivos tokenizados
1500/25000 archivos tokenizados
1600/25000 archivos tokenizados
1700/25000 archivos tokenizados
1800/25000 archivos tokenizados
1900/25000 archivos tokenizados
2000/25000 archivos tokenizados
2100/25000 archivos tokenizados
2200/25000 archivos tokenizados
2300/25000 archivos tokenizados
2400/25000 archivos tokenizados
2500/25000 archivos tokenizados
2600/25000 archivos tokenizados
2700/25000 archivos tokenizados
2800/25000 archivos tokenizados
2900/25000 archivos tokenizados
3000/25000 archivos tokenizados
3100/25000 archivos tokenizados
3200/25000 archiv

In [ ]:
test_tokens

,chunk_id,token_ids,num_tokens,uid,pid
0,0,"[101, 1013, 1013, 1013, 2065, 1045, 2156, 2008...",285,NaN,34841
1,0,"[101, 1013, 1008, 7953, 1015, 1015, 1008, 1013...",512,NaN,68320
2,1,"[101, 1004, 12098, 2290, 2487, 1010, 12098, 56...",512,NaN,68320
3,2,"[101, 2522, 4904, 1026, 1026, 1015, 1026, 1026...",112,NaN,68320
4,0,"[101, 1001, 2421, 1026, 9896, 1028, 1001, 2421...",512,NaN,72734
...,...,...,...,...,...
47570,0,"[101, 1001, 2421, 1026, 9017, 1013, 2358, 1640...",87,NaN,62563
47571,0,"[101, 1001, 2421, 1026, 9017, 1013, 2358, 1640...",512,NaN,45920
47572,1,"[101, 1007, 1010, 25022, 2078, 1012, 5495, 100...",377,NaN,45920
47573,0,"[101, 1001, 2421, 1026, 9017, 1013, 2358, 1640...",192,NaN,81447


In [ ]:
test_tokens.to_csv("test_tokens.csv", index=False)

print(f"Guardado en: {output_path}")

Guardado en: /content/drive/MyDrive/IA\ y\ Compiladores/Reto_plagio/dataset/train_source.parquet
